In [ ]:
import lightning as L
import torch
from IPython.display import Video

from training.evaluate import evaluate, record_video
from training.ppo import PPO, ActorCritic, PPOData
from training.sim import SKILLS, DuckEnv, hardware


In [ ]:
hw = hardware.load("2026-11-07-hd1910")
env = DuckEnv(SKILLS["walk"], hw, num_envs=2048, device="cuda", seed=1)
env.reset()
env.randomize_episode_starts()                           # 回合进度打散，免得 2048 只在同一步一起超时
net = ActorCritic(env.obs_dim, env.critic_obs_dim, env.action_dim).to("cuda")
net

In [ ]:
data = PPOData(env, net)      # data：每轮用当前网络采 24 步 → 算优势 → 切成 4 个小批、用 5 遍
model = PPO(net)              # model：一个小批的损失，更新网络

In [ ]:
batches = data.batches(data.prepare(data.collect()))
len(batches), {key: tuple(value.shape) for key, value in batches[0].items()}

In [ ]:
before = evaluate(net, SKILLS["walk"], hw)
before["falls"], round(before["mean_step_reward"], 3)

In [ ]:
trainer = L.Trainer(max_epochs=300, reload_dataloaders_every_n_epochs=1,    # 每轮按最新的网络重新采样
                    accelerator="gpu", devices=1, logger=False, enable_checkpointing=False)
trainer.fit(model, data)

In [ ]:
after = evaluate(net, SKILLS["walk"], hw)
after["falls"], round(after["mean_step_reward"], 3)

In [ ]:
record_video(net, SKILLS["walk"], hw, "m04_walk.mp4")
Video("m04_walk.mp4", embed=True)

In [ ]:
torch.save(net.state_dict(), "m04_walk.pt")